In [2]:
# Install gdown and download from Google Drive
# !pip install -q gdown
# !gdown 1df8ZwYCPWZczJs1mX8akslow_NAJfZ7B

Downloading...
From: https://drive.google.com/uc?id=1df8ZwYCPWZczJs1mX8akslow_NAJfZ7B
To: /home/azureuser/localfiles/cortex-project/v1_text_data.jsonl
100%|███████████████████████████████████████| 12.2M/12.2M [00:00<00:00, 154MB/s]
100%|███████████████████████████████████████| 12.2M/12.2M [00:00<00:00, 154MB/s]


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1,2,3,5,6"
DOWNLOAD_PATH = "datasets/VisualQA-PMCArticle-Dataset/train"
os.makedirs(DOWNLOAD_PATH, exist_ok=True)

In [2]:
DATASET_ROOT_DIR = "datasets/VisualQA-PMCArticle-Dataset"

### News

### Installation

In [6]:
# %%capture
# import os, re
# if "COLAB_" not in "".join(os.environ.keys()):
#     !pip install unsloth
# else:
#     # Do this only in Colab notebooks! Otherwise use pip install unsloth
#     import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
#     xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
#     !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
#     !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
#     !pip install --no-deps unsloth
# !pip install transformers==4.56.2
# !pip install --no-deps trl==0.22.2

### Unsloth

In [3]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit", # Llama 3.2 vision support
    "unsloth/Llama-3.2-11B-Vision-bnb-4bit",
    "unsloth/Llama-3.2-90B-Vision-Instruct-bnb-4bit", # Can fit in a 80GB card!
    "unsloth/Llama-3.2-90B-Vision-bnb-4bit",

    "unsloth/Pixtral-12B-2409-bnb-4bit",              # Pixtral fits in 16GB!
    "unsloth/Pixtral-12B-Base-2409-bnb-4bit",         # Pixtral base model

    "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",          # Qwen2 VL support
    "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit",
    "unsloth/Qwen2-VL-72B-Instruct-bnb-4bit",

    "unsloth/llava-v1.6-mistral-7b-hf-bnb-4bit",      # Any Llava variant works!
    "unsloth/llava-1.5-7b-hf-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Llama-3.2-11B-Vision-Instruct",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 12-08 18:56:02 [__init__.py:239] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!


/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.1.1) and mlflow-skinny (2.22.1) are different. This may lead to unexpected behavior. Please install the same version of both packages.
  mlflow.mismatch._check_version_mismatch()


==((====))==  Unsloth 2025.12.1: Fast Mllama patching. Transformers: 4.56.2. vLLM: 0.8.4.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.151 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/477 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

**[NEW]** We also support finetuning ONLY the vision part of the model, or ONLY the language part. Or you can select both! You can also select to finetune the attention or the MLP layers!

In [4]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

Unsloth: Making `model.base_model.model.model.vision_model.transformer` require gradients


<a name="Data"></a>
### Image Data Prep

In [ ]:
!gdown --folder 1jVE2sR69Cf8zVJTl0Xlxzr80bGIEItV


usage: gdown [-h] [-V] [-O OUTPUT] [-q] [--fuzzy] [--id] [--proxy PROXY]
             [--speed SPEED] [--no-cookies] [--no-check-certificate]
             [--continue] [--folder] [--remaining-ok] [--format FORMAT]
             [--user-agent USER_AGENT]
             url_or_id
gdown: error: unrecognized arguments: train/llava_med_alignment_500k.json


In [17]:
df_img_align = pd.read_json(f'{DATASET_ROOT_DIR}/train/llava_med_alignment_500k.json')

In [18]:
import os
all_images = os.listdir(f"{DATASET_ROOT_DIR}/train/images")

In [19]:
df_img_align["image"] = df_img_align.apply(lambda r: os.path.join(f"{DATASET_ROOT_DIR}/train/images/", r["image"]) if r["image"] in all_images else None, axis=1)

In [20]:
df_img_align = df_img_align.dropna()
df_img_align.shape

(138, 3)

In [21]:
df_img_align.to_json(f"{DATASET_ROOT_DIR}/train/v1_image_data.jsonl", orient="records", lines=True)

In [22]:
df_img_align

,id,image,conversatons
2220,30241470_Fig3,datasets/VisualQA-PMCArticle-Dataset/train/ima...,"[{'from': 'human', 'value': 'Write an exhausti..."
3656,29112170_molecules-22-01896-f005,datasets/VisualQA-PMCArticle-Dataset/train/ima...,"[{'from': 'human', 'value': 'Portray the image..."
3821,30241472_Fig2,datasets/VisualQA-PMCArticle-Dataset/train/ima...,"[{'from': 'human', 'value': 'Provide a detaile..."
4599,29156571_molecules-22-02007-f016,datasets/VisualQA-PMCArticle-Dataset/train/ima...,"[{'from': 'human', 'value': 'Give an elaborate..."
9761,30224556_LM047753JIAF5,datasets/VisualQA-PMCArticle-Dataset/train/ima...,"[{'from': 'human', 'value': 'Examine the image..."
...,...,...,...
454013,29140310_molecules-22-01982-f002,datasets/VisualQA-PMCArticle-Dataset/train/ima...,"[{'from': 'human', 'value': 'Give a short and ..."
461838,30241539_Fig2,datasets/VisualQA-PMCArticle-Dataset/train/ima...,"[{'from': 'human', 'value': 'Analyze the image..."
462924,29140285_molecules-22-01975-f003,datasets/VisualQA-PMCArticle-Dataset/train/ima...,"[{'from': 'human', 'value': 'Summarize the vis..."
463903,30271138_f7-ijn-13-5419,datasets/VisualQA-PMCArticle-Dataset/train/ima...,"[{'from': 'human', 'value': 'Summarize the vis..."


In [23]:
df_img_align_dict = df_img_align.rename(columns={"conversatons":"conversations"}).to_dict(orient="list")

In [24]:
from datasets import Dataset, Image
dataset = Dataset.from_dict(df_img_align_dict).cast_column("image", Image())

In [25]:
dataset[2]

{'id': '30241472_Fig2',
 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=L size=596x1137>,
 'conversations': [{'from': 'human',
   'value': 'Provide a detailed description of the given image\n<image>'},
  {'from': 'gpt',
   'value': 'Effect of different temperatures (a), pH (b), and proteinase K digestion (c) on flocculation activity of the bioflocculant. All error bars indicate the SE of the three biological replicates. *represents a statistically significant difference of p < 0.05 compared with positive control'}]}

In [26]:
h = 0
g = 0
c = 0
for r in dataset:
  source = [c["from"] for c in r["conversations"]]
  if "human" in source:
    h += 1
  if "gpt" in source:
    g += 1
  if "human" in source and "gpt" in source:
    c += 1

print(h, g, c)

138 138 138


To format the dataset, all vision finetuning tasks should be formatted as follows:

```python
[
{ "role": "user",
  "content": [{"type": "text",  "text": instruction}, {"type": "image", "image": image} ]
},
{ "role": "assistant",
  "content": [{"type": "text",  "text": answer} ]
},
]
```

We will craft an custom instruction asking the VLM to be an expert radiographer. Notice also instead of just 1 instruction, you can add multiple turns to make it a dynamic conversation.

## Text Data Prep

In [27]:
import pandas as pd

t1 = pd.read_json(f"{DATASET_ROOT_DIR}/model2_processed_reasoning_no_regenerate_tools.jsonl", lines=True)
t2 = pd.read_json(f"{DATASET_ROOT_DIR}/model2_processed_reasoning_with_regenerate_tools.jsonl", lines=True)

In [28]:
t2.metadata[0]

{'question': 'A 35-year-old woman comes to the physician because of a 1-month history of double vision, difficulty climbing stairs, and weakness when trying to brush her hair. She reports that these symptoms are worse after she exercises and disappear after she rests for a few hours. Physical examination shows drooping of her right upper eyelid that worsens when the patient is asked to gaze at the ceiling for 2 minutes. There is diminished motor strength in the upper extremities. The remainder of the examination shows no abnormalities. Which of the following is the most likely diagnosis?\nOptions: Myasthenia gravis / Polymyositis / Amyotrophic lateral sclerosis / Guillain-Barré syndrome / Multiple sclerosis',
 'regenerated_tool_responses': []}

In [29]:
import json
t2["filter"] = t2.apply(lambda r: len(r["metadata"]["regenerated_tool_responses"]) > 0 and all(c != "null" for c in r["metadata"]["regenerated_tool_responses"]), axis=1)
t2 = t2[t2["filter"]]

In [30]:
t2.columns

Index(['instruction', 'input', 'output', 'ground_truth_answer',
       'with_tool_info', 'metadata', '_dp_index', 'filter'],
      dtype='object')

In [31]:
t2.shape

(18, 8)

In [32]:
t_final = pd.concat([t1, t2.drop(columns="filter")], axis=0).drop(columns="_dp_index")
t_final.describe()


,instruction,input,output,ground_truth_answer,with_tool_info,metadata
count,1710,1710,1710,1710,1710,1710
unique,1,1705,1692,1540,2,1705
top,Based on the medical question and tool results...,"## TOOL RESULT\n[""{\""error\"": \""Unsupported di...",Von Willebrand disease. The patient's symptom...,Allopurinol,False,{'question': 'A 5-year-old girl is brought to ...
freq,1710,2,3,5,951,2


In [33]:
t_final.with_tool_info.value_counts()

with_tool_info
False    951
True     759
Name: count, dtype: int64

In [34]:
t_final["conversations"] = t_final.apply(lambda r : [{"from": "human", "value": r["input"]},{"from": "gpt", "value": r["output"]+f"\n## ANSWER\n{r['ground_truth_answer']}"}], axis=1)

In [35]:
t_final["image"] = None
t_final["id"] = t_final.apply(lambda r: f"text-tool_{r['with_tool_info']}-{r.name}", axis=1)
t_final_dict = t_final[["image", "conversations", "id"]].to_dict(orient="list")
t_final[["image", "conversations", "id"]].to_json(f"{DATASET_ROOT_DIR}/train/v1_text_data.jsonl", lines=True, orient="records")

In [36]:
t_final["output_old"] = t_final["output"]
t_final_

NameError: name 't_final_' is not defined

In [ ]:
t_final

,instruction,input,output,ground_truth_answer,with_tool_info,metadata,conversations,image,id
0,Based on the medical question and tool results...,"## TOOL RESULT\n"":\n[FDA_get_indications_by_dr...",Nitrofurantoin is the best treatment for this...,Nitrofurantoin,True,{'question': 'A 23-year-old pregnant woman at ...,"[{'from': 'human', 'value': '## TOOL RESULT "":...",None,text-tool_True-0
1,Based on the medical question and tool results...,"## TOOL RESULT\n"":\n[kegg_search_pathway(keywo...","The symptoms described, such as regurgitation...",Abnormal migration of ventral pancreatic bud,False,{'question': 'A mother brings her 3-week-old i...,"[{'from': 'human', 'value': '## TOOL RESULT "":...",None,text-tool_False-1
2,Based on the medical question and tool results...,"## TOOL RESULT\n"":\n[clinvar_search_variants(c...",Von Willebrand disease. The patient's symptom...,Von Willebrand disease,True,{'question': 'A 20-year-old woman presents wit...,"[{'from': 'human', 'value': '## TOOL RESULT "":...",None,text-tool_True-2
3,Based on the medical question and tool results...,"## TOOL RESULT\n"":\n[MPD_get_phenotype_data(li...",The congenital diaphragmatic defect in the pl...,Gastric fundus in the thorax,False,{'question': 'A 3900-g (8.6-lb) male infant is...,"[{'from': 'human', 'value': '## TOOL RESULT "":...",None,text-tool_False-3
4,Based on the medical question and tool results...,"## TOOL RESULT\n"":\n[FDA_get_indications_by_dr...",Digoxin is the best choice for rate control i...,Digoxin,True,{'question': 'A 62-year-old woman presents for...,"[{'from': 'human', 'value': '## TOOL RESULT "":...",None,text-tool_True-4
...,...,...,...,...,...,...,...,...,...
505,Based on the medical question and tool results...,"## TOOL RESULT\n[""[{\""title\"": \""Electrical st...",The most likely underlying cause of the patie...,Progressive obstruction of expiratory airflow,False,{'question': 'A 62-year-old woman comes to the...,"[{'from': 'human', 'value': '## TOOL RESULT [""...",None,text-tool_False-505
506,Based on the medical question and tool results...,"## TOOL RESULT\n[""{\""error\"": \""Unsupported di...",The most likely diagnosis for the 5-year-old ...,Granulosa cell tumor,False,{'question': 'A 5-year-old girl is brought to ...,"[{'from': 'human', 'value': '## TOOL RESULT [""...",None,text-tool_False-506
509,Based on the medical question and tool results...,"## TOOL RESULT\n[""{\""meta\"": {\""skip\"": 0, \""l...",The medication most likely to cause the sympt...,Cyclophosphamide,True,{'question': 'A 70-year-old man presents to a ...,"[{'from': 'human', 'value': '## TOOL RESULT [""...",None,text-tool_True-509
510,Based on the medical question and tool results...,"## TOOL RESULT\n[""{\""error\"": \""Parameter vali...",The most likely structure destroyed by the vi...,Anterior horn of the spinal cord,False,{'question': 'A 12-year-old boy who recently e...,"[{'from': 'human', 'value': '## TOOL RESULT [""...",None,text-tool_False-510


## Combine Image and Text Data

In [40]:
import pandas as pd
df_img_align = pd.read_json(f"{DATASET_ROOT_DIR}/train/v2_image_data.jsonl", lines=True)
# t_final = pd.read_json(f"{DATASET_ROOT_DIR}/train/v1_text_data.jsonl", lines=True)
# t_final["image"] = None

In [38]:
final_image_text_df

NameError: name 'final_image_text_df' is not defined

In [39]:
# final_image_text_df = pd.concat([df_img_align.rename(columns={"conversatons":"conversations"}), t_final], axis=0)

# Using only images for training
final_image_text_df = df_img_align.rename(columns={"conversatons":"conversations"})

from sklearn.model_selection import train_test_split
train, test, _, _ = train_test_split(
    final_image_text_df, final_image_text_df, test_size=0.25, random_state=42
)

train_dict = train.to_dict(orient="list")
test_dict = test.to_dict(orient="list")

In [41]:
from datasets import Dataset, Image, DatasetDict

dataset_train = Dataset.from_dict(train_dict).cast_column("image", Image())
dataset_test = Dataset.from_dict(test_dict).cast_column("image", Image())

# dataset = Dataset.from_dict(final_image_text_df_dict).cast_column("image", Image())

In [42]:
SYSTEM_PROMPT_REASON = """You are a medical reasoning assistant. Your goal is to determine the single best answer to a medical question using the provided tool results (e.g., clinical guidelines, studies, drug information, regulatory documents).

Your responsibilities:
1. Carefully read the question and all tool outputs.
2. Base your reasoning explicitly on the provided evidence. Do not invent facts or draw on external knowledge unless needed for basic medical concepts.
3. When explaining your reasoning, clearly reference the specific tool results that support your conclusion.
4. If a question requires selecting one best option, choose only one.
5. If the question is open-ended, provide a concise evidence-based answer.

Output format:
- Provide your full reasoning first.
- On a new line at the end of your response, include the final answer in this exact format if you are selecting an answer:

## ANSWER
<FINAL-ANSWER>

(Replace <FINAL-ANSWER> with the chosen option or final short answer.)
"""

In [43]:
def convert_to_conversation(sample):
    source = {c["from"]:c["value"] for c in sample["conversations"]}
    conversation = [{
      "role": "system",
      "content": [{
          "type": "text", "text": SYSTEM_PROMPT_REASON
      }]
    }] if sample['image'] is None else []
    user_content = [
      {"type" : "text",  "text"  : SYSTEM_PROMPT_REASON+source["human"]},
      {"type" : "image", "image" : sample["image"]}
    ] if sample["image"] else [{"type" : "text",  "text"  : source["human"]}]
    conversation.extend([
        { "role": "user",
          "content" : user_content
        },
        { "role" : "assistant",
          "content" : [
            {"type" : "text",  "text"  : source["gpt"]} ]
        },
    ])
    return { "messages" : conversation }
pass

Let's convert the dataset into the "correct" format for finetuning:

In [44]:
converted_train_dataset = [convert_to_conversation(sample) for sample in dataset_train]
converted_test_dataset = [convert_to_conversation(sample) for sample in dataset_test]

The first example is now structured like below:

In [45]:
# train image sample
converted_train_dataset[0]

{'messages': [{'role': 'user',
   'content': [{'type': 'text',
     'text': 'You are a medical reasoning assistant. Your goal is to determine the single best answer to a medical question using the provided tool results (e.g., clinical guidelines, studies, drug information, regulatory documents).\n\nYour responsibilities:\n1. Carefully read the question and all tool outputs.\n2. Base your reasoning explicitly on the provided evidence. Do not invent facts or draw on external knowledge unless needed for basic medical concepts.\n3. When explaining your reasoning, clearly reference the specific tool results that support your conclusion.\n4. If a question requires selecting one best option, choose only one.\n5. If the question is open-ended, provide a concise evidence-based answer.\n\nOutput format:\n- Provide your full reasoning first.\n- On a new line at the end of your response, include the final answer in this exact format if you are selecting an answer:\n\n## ANSWER\n<FINAL-ANSWER>\n\n(

In [46]:
# test image sample
converted_test_dataset[0]

{'messages': [{'role': 'user',
   'content': [{'type': 'text',
     'text': 'You are a medical reasoning assistant. Your goal is to determine the single best answer to a medical question using the provided tool results (e.g., clinical guidelines, studies, drug information, regulatory documents).\n\nYour responsibilities:\n1. Carefully read the question and all tool outputs.\n2. Base your reasoning explicitly on the provided evidence. Do not invent facts or draw on external knowledge unless needed for basic medical concepts.\n3. When explaining your reasoning, clearly reference the specific tool results that support your conclusion.\n4. If a question requires selecting one best option, choose only one.\n5. If the question is open-ended, provide a concise evidence-based answer.\n\nOutput format:\n- Provide your full reasoning first.\n- On a new line at the end of your response, include the final answer in this exact format if you are selecting an answer:\n\n## ANSWER\n<FINAL-ANSWER>\n\n(

In [47]:
# 3. Combine into a DatasetDict
dataset = DatasetDict({
    'train': converted_train_dataset,
    'test': dataset_test,
})

Before we do any finetuning, maybe the vision model already knows how to analyse the images? Let's check if this is the case!

In [48]:
FastVisionModel.for_inference(model) # Enable for inference!

# Vision inference
image = dataset_test[0]["image"]

messages = converted_test_dataset[0]["messages"]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

## Step Step 1
1
**Question **Question Analysis**: Analysis**: This This question question requires requires us us to to evaluate evaluate two two chemicals chemicals based based on on their their structure.

structure.

## ## Step Step 2: 2: 1,10-Phenanthroline
1,10-Phenanthroline
1,10-Phenanthroline, 1,10-Phenanthroline, as as indicated indicated in in the the chemical chemical structure structure labeled labeled as as 1 1 in in the the diagram, diagram, is is a a nitrogen-containing nitrogen-containing aromatic aromatic compound. compound. It It is is primarily primarily used used as as a a reagent reagent for for identifying identifying heavy heavy metals.

metals.

## ## Step Step 3: 3: Neocuproine Neocuproine HCl
HCl
Neocuproine, Neocuproine, labeled labeled as as 2 2 in in the the diagram, diagram, has has a a similar similar structure structure to to 1,10-phenanthroline 1,10-phenanthroline but but is is a a compound compound more more commonly used
commonly used


In [ ]:
# FastVisionModel.for_inference(model) # Enable for inference!

# # Text inference
# messages = converted_dataset[-1]["messages"]
# input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
# inputs = tokenizer(
#     None,
#     input_text,
#     add_special_tokens = False,
#     return_tensors = "pt",
# ).to("cuda")

# from transformers import TextStreamer
# text_streamer = TextStreamer(tokenizer, skip_prompt = True)
# _ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
#                    use_cache = True, temperature = 1.5, min_p = 0.1)

The symptoms presented by the 23-year-old patient, such as sudden and increased vaginal bleeding, the presence of clots, and the visualization of products of conception in the os, are highly indicative of a miscarriage. These symptoms are not strongly associated with sexually transmitted diseases, Rh immunization, antiphospholipid syndrome, or trauma.

Furthermore, chromosomal abnormalities are a well-documented cause of miscarriage, especially in the first trimester. This condition often results from an error in the replication of genetic material during cell division, leading to an abnormality in the chromosomal structure of the developing embryo.

While antiphosph


<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

We use our new `UnslothVisionDataCollator` which will help in our vision finetuning setup.

In [53]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model) # Enable for training!

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer), # Must use!
    train_dataset = converted_train_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,
        # num_train_epochs = 1, # Set this instead of max_steps for full training runs
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",     # For Weights and Biases

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    ),
)

In [54]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-80GB. Max memory = 79.151 GB.
9.873 GB of memory reserved.


In [55]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 103 | Num Epochs = 3 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 67,174,400 of 10,737,395,235 (0.63% trained)


Step,Training Loss
1,4.033400
2,3.799000
3,3.252000
4,2.739900
5,2.624200
6,2.183100
7,2.044600
8,1.633600
9,1.438100
10,1.110700


In [56]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

169.8458 seconds used for training.
2.83 minutes used for training.
Peak reserved memory = 10.516 GB.
Peak reserved memory for training = 0.643 GB.
Peak reserved memory % of max memory = 13.286 %.
Peak reserved memory for training % of max memory = 0.812 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

We use `min_p = 0.1` and `temperature = 1.5`. Read this [Tweet](https://x.com/menhguin/status/1826132708508213629) for more information on why.

In [57]:
FastVisionModel.for_inference(model) # Enable for inference!

# Vision inference
image = dataset_test[0]["image"]

messages = converted_test_dataset[0]["messages"]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

Nuclear structures structures of of (a) (a) 1,10-phenanthroline 1,10-phenanthroline and and (b) (b) neocuproine.<|eot_id|>
neocuproine.<|eot_id|>


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [60]:
dir = "checkpoints/M2Model/Llama"
os.makedirs(f"{dir}/train/lora_model/v2/")
model.save_pretrained(f"{dir}/train/lora_model/v2/")  # Local saving
tokenizer.save_pretrained(f"{dir}/train/lora_model/v2/")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

[]

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [62]:
if False:
    from unsloth import FastVisionModel
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = True, # Set to False for 16bit LoRA
    )
    FastVisionModel.for_inference(model) # Enable for inference!

# Use dataset_test instead of dataset[0]
image = dataset_test[0]["image"]
instruction = "You are an expert radiographer. Describe accurately what you see in this image."

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

Two chemical chemical structures structures of of benzocaine benzocaine (1) (1) and and 1-methylbenzocaine 1-methylbenzocaine (2) (2) are are depicted depicted with with two two stereocenters stereocenters each, each, labeled labeled as as N N and and CH3.

CH3.

N, N, or or the the amine amine nitrogen nitrogen atom, atom, is is attached attached to to the the first first stereocenter stereocenter through through a a two-carbon two-carbon chain chain (CH2 (CH2 group). group). The The first first stereocenter stereocenter is is attached attached to to an an aromatic aromatic ring ring via via two two bonds; bonds; a a second second carbon carbon (C2) (C2) attached attached to to the the ring ring forms forms the the second second bond, bond, and and a a second second carbon carbon (C3) (C3) attached attached to to N N and and the the first first stereocenter stereocenter form form a a third third bond bond to to the the ring. ring. C2 C2 is is also connected
also connected


### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Select ONLY 1 to save! (Both not needed!)

# Save locally to 16bit
if False: model.save_pretrained_merged("unsloth_finetune", tokenizer,)

# To export and save to your Hugging Face account
if False: model.push_to_hub_merged("YOUR_USERNAME/unsloth_finetune", tokenizer, token = "PUT_HERE")

In [ ]:
# Select ONLY 1 to save! (Both not needed!)

# Save locally to 16bit
if False: model.save_pretrained_merged("unsloth_finetune", tokenizer,)

# To export and save to your Hugging Face account
if False: model.push_to_hub_merged("YOUR_USERNAME/unsloth_finetune", tokenizer, token = "PUT_HERE")